In [1]:
import pandas as pd
df = pd.read_parquet("../data/fct_prices_hourly.parquet")
df.shape

(24527, 10)

In [2]:
copy_df = df.sort_values(by = "ts_utc")
copy_df["naive_pred"] = copy_df["price_eur_mwh"].shift(168)
copy_df.head(5)

,ts_utc,ts_berlin,price_eur_mwh,date_berlin,hour_berlin,day_of_week,month,is_weekend,is_holiday,is_day_off,naive_pred
5981,2023-12-31 23:00:00+00:00,2024-01-01 00:00:00,0.10,2024-01-01,0,1,1,False,True,True,NaN
12190,2024-01-01 00:00:00+00:00,2024-01-01 01:00:00,0.01,2024-01-01,1,1,1,False,True,True,NaN
21377,2024-01-01 01:00:00+00:00,2024-01-01 02:00:00,0.00,2024-01-01,2,1,1,False,True,True,NaN
9156,2024-01-01 02:00:00+00:00,2024-01-01 03:00:00,-0.01,2024-01-01,3,1,1,False,True,True,NaN
18259,2024-01-01 03:00:00+00:00,2024-01-01 04:00:00,-0.03,2024-01-01,4,1,1,False,True,True,NaN


In [3]:
copy_df.info()

<class 'pandas.DataFrame'>
Index: 24527 entries, 5981 to 18235
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   ts_utc         24527 non-null  datetime64[us, UTC]
 1   ts_berlin      24527 non-null  datetime64[us]     
 2   price_eur_mwh  24527 non-null  float64            
 3   date_berlin    24527 non-null  object             
 4   hour_berlin    24527 non-null  int64              
 5   day_of_week    24527 non-null  int64              
 6   month          24527 non-null  int64              
 7   is_weekend     24527 non-null  bool               
 8   is_holiday     24527 non-null  bool               
 9   is_day_off     24527 non-null  bool               
 10  naive_pred     24359 non-null  float64            
dtypes: bool(3), datetime64[us, UTC](1), datetime64[us](1), float64(2), int64(3), object(1)
memory usage: 1.8+ MB


In [4]:
son_tarih = copy_df["ts_utc"].max()
test = copy_df[copy_df["ts_utc"] > son_tarih - pd.Timedelta(days=365)]
test

,ts_utc,ts_berlin,price_eur_mwh,date_berlin,hour_berlin,day_of_week,month,is_weekend,is_holiday,is_day_off,naive_pred
5199,2025-10-18 22:00:00+00:00,2025-10-19 00:00:00,92.3425,2025-10-19,0,7,10,True,False,True,96.1775
8278,2025-10-18 23:00:00+00:00,2025-10-19 01:00:00,87.5850,2025-10-19,1,7,10,True,False,True,92.1675
17431,2025-10-19 00:00:00+00:00,2025-10-19 02:00:00,82.2925,2025-10-19,2,7,10,True,False,True,88.8600
20526,2025-10-19 01:00:00+00:00,2025-10-19 03:00:00,82.1400,2025-10-19,3,7,10,True,False,True,85.1300
23690,2025-10-19 02:00:00+00:00,2025-10-19 04:00:00,83.2125,2025-10-19,4,7,10,True,False,True,86.4125
...,...,...,...,...,...,...,...,...,...,...,...
9123,2026-10-18 17:00:00+00:00,2026-10-18 19:00:00,220.5975,2026-10-18,19,7,10,True,False,True,234.7875
21362,2026-10-18 18:00:00+00:00,2026-10-18 20:00:00,198.2750,2026-10-18,20,7,10,True,False,True,229.1200
18257,2026-10-18 19:00:00+00:00,2026-10-18 21:00:00,179.5475,2026-10-18,21,7,10,True,False,True,214.4675
24457,2026-10-18 20:00:00+00:00,2026-10-18 22:00:00,176.6075,2026-10-18,22,7,10,True,False,True,198.9350


In [5]:
mae = (test["price_eur_mwh"] - test["naive_pred"]).abs().mean()
print(f"MAE: {mae}")

MAE: 38.48019434931507


In [6]:
def evaluate_decisions(df, pred_col):
    d = df.sort_values("ts_utc").copy()

    # Sadece tam günleri al (23, 24 veya 25 saat)
    gun_saat = d.groupby("date_berlin")["ts_utc"].transform("count")
    d = d[gun_saat >= 23]

    # Her saat için: bu saatte BİTEN 6 saatlik bloğun toplamı (gün içinde)
    grp = d.groupby("date_berlin")
    d["pred_block"] = grp[pred_col].transform(lambda s: s.rolling(6).sum())
    d["actual_block"] = grp["price_eur_mwh"].transform(lambda s: s.rolling(6).sum())

    # Model: tahmine göre en ucuz görünen bloğu seç, gerçek maliyetini al
    secilen = d.loc[d.groupby("date_berlin")["pred_block"].idxmin()]
    model_cost = secilen.set_index("date_berlin")["actual_block"]

    # Kusursuz bilgi: gerçek fiyatlara göre en ucuz blok
    oracle_cost = d.groupby("date_berlin")["actual_block"].min()

    # Kural: 10-15 saatlerinin toplamı
    rule_cost = d[d["hour_berlin"].between(10, 15)].groupby("date_berlin")["price_eur_mwh"].sum()

    toplam_rule, toplam_oracle, toplam_model = rule_cost.sum(), oracle_cost.sum(), model_cost.sum()
    yakalama = (toplam_rule - toplam_model) / (toplam_rule - toplam_oracle)

    print(f"Kural maliyeti:     {toplam_rule:,.0f} €")
    print(f"Model maliyeti:     {toplam_model:,.0f} €")
    print(f"Kusursuz maliyet:   {toplam_oracle:,.0f} €")
    print(f"Yakalama oranı:     {yakalama:.1%}")
    return yakalama


In [7]:
print(evaluate_decisions(test, "naive_pred"))

Kural maliyeti:     139,102 €
Model maliyeti:     124,962 €
Kusursuz maliyet:   116,526 €
Yakalama oranı:     62.6%
0.6263375938410031


In [8]:
features = ["hour_berlin", "day_of_week", "month", "is_day_off", "naive_pred"]

In [9]:
print(f"Droplamadan önce satır sayısı: {len(copy_df)}")
copy_df = copy_df.dropna(subset=["naive_pred"])
print(f"Dropladıktan sonra satır sayısı: {len(copy_df)}")

Droplamadan önce satır sayısı: 24527
Dropladıktan sonra satır sayısı: 24359


In [13]:
copy_df["date_berlin"] = pd.to_datetime(copy_df["date_berlin"], format="%Y-%m-%d")
print(type(copy_df["date_berlin"])) # dönüşümün başarılı olduğunu kontrol ediyoruz.

<class 'pandas.Series'>


In [18]:
copy_df.info()

<class 'pandas.DataFrame'>
Index: 24359 entries, 9240 to 18235
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   ts_utc         24359 non-null  datetime64[us, UTC]
 1   ts_berlin      24359 non-null  datetime64[us]     
 2   price_eur_mwh  24359 non-null  float64            
 3   date_berlin    24359 non-null  datetime64[s]      
 4   hour_berlin    24359 non-null  int64              
 5   day_of_week    24359 non-null  int64              
 6   month          24359 non-null  int64              
 7   is_weekend     24359 non-null  bool               
 8   is_holiday     24359 non-null  bool               
 9   is_day_off     24359 non-null  bool               
 10  naive_pred     24359 non-null  float64            
dtypes: bool(3), datetime64[s](1), datetime64[us, UTC](1), datetime64[us](1), float64(2), int64(3)
memory usage: 1.7 MB


In [22]:
# Train / Test bölmesi
train = copy_df[copy_df["date_berlin"] <= (son_tarih - pd.Timedelta(days=365)).tz_localize(None)]
test_yeni = copy_df[copy_df["date_berlin"] > (son_tarih - pd.Timedelta(days=365)).tz_localize(None)]
print(f"Train gözlem sayısı: {train.shape[0]}, Test gözlem sayısı: {test_yeni.shape[0]}")

Train gözlem sayısı: 15599, Test gözlem sayısı: 8760


In [24]:
X_train, y_train, X_test, y_test = train[features], train["price_eur_mwh"], test_yeni[features], test_yeni["price_eur_mwh"]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (15599, 5), y_train shape: (15599,)


In [25]:
from sklearn.ensemble import HistGradientBoostingRegressor
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train) # sorular ve cevaplarıyla modeli eğitiyoruz.
test_yeni["model_pred"] = model.predict(X_test) # eğitim sonucu modelin başarısını ölçmek için modele soruları verelim.
MAE = (test_yeni["price_eur_mwh"] - test_yeni["model_pred"]).abs().mean()
print(f"MAE: {MAE}")
evaluate_decisions(test_yeni, "model_pred")

MAE: 38.27947303900178
Kural maliyeti:     139,102 €
Model maliyeti:     123,681 €
Kusursuz maliyet:   116,526 €
Yakalama oranı:     68.3%


np.float64(0.6830456703491938)